In [1]:
# Cell 1: Imports and Global Config
%reload_ext autoreload
%autoreload 2

import os
import sys
import random
import traceback

import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP

# Insert alignment_v2 path if needed, for example:
# sys.path.insert(0, "/path/to/alignment_v2")

from alignment_v2.datasets import get_dataset
from alignment_v2.models.registry import get_model
from alignment_v2 import train
from alignment_v2.train import progressive_dropout
from alignment_v2 import plotting

# Toggle this to choose DDP or single-process
USE_DDP = True

# Basic hyperparams
WORLD_SIZE = 4  # Number of GPUs on the node
EPOCHS = 5
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
DROPOUT_RATE = 0.0

model_name = "MLP"
dataset_name = "MNIST"

In [2]:
# Cell 2: The DDP Worker Function
def ddp_worker(rank, world_size, port, epochs, batch_size, lr, dropout_rate):
    """
    Each spawned process calls this function if we're using DDP mode.
    """
    try:
        master_addr = "127.0.0.1"
        os.environ["MASTER_ADDR"] = master_addr
        os.environ["MASTER_PORT"] = str(port)
        dist.init_process_group("nccl", rank=rank, world_size=world_size)

        torch.cuda.set_device(rank)
        device = torch.device(f"cuda:{rank}")

        print(f"\n[DDP_RANK={rank}] world_size={world_size}, port={port}, device={device}")

        net = get_model(
            model_name,
            build=True,
            dataset=dataset_name,
            dropout=dropout_rate,
            ignore_flag=False
        )
        net.to(device)

        ddp_net = DDP(net, device_ids=[rank], output_device=rank)

        loader_params = dict(batch_size=batch_size, shuffle=False, num_workers=4)
        dataset = get_dataset(
            dataset_name,
            build=True,
            transform_parameters=ddp_net.module,
            loader_parameters=loader_params,
            device="cuda",
            distributed=True
        )

        print(f"[DDP_RANK={rank}] Dataset ready. Creating optimizer...")

        optimizer = torch.optim.Adam(ddp_net.parameters(), lr=lr, weight_decay=0)

        train_params = dict(
            num_epochs=epochs,
            alignment=True,
            alignment_expansion=False,
            compare_expected=False,
            frequency=1,
            delta_alignment=False,
        )
        nets = [ddp_net]
        optimizers = [optimizer]

        print(f"[DDP_RANK={rank}] Starting training for {epochs} epochs...")
        results = train.train(nets, optimizers, dataset, **train_params)
        print(f"[DDP_RANK={rank}] Done training. Results keys={list(results.keys())}")

        if rank == 0:
            print("[DDP_RANK=0] Doing progressive dropout experiment.")
            if "alignment" not in results:
                print("[DDP_RANK=0] 'alignment' not found => skipping dropout.")
            else:
                dropout_params = dict(
                    num_drops=3,
                    by_layer=False,
                    train_set=False,
                )
                drop_res = progressive_dropout(nets, dataset, alignment=results["alignment"], **dropout_params)
                print("[DDP_RANK=0] dropout results keys:", list(drop_res.keys()))
                plotting.plot_dropout_results(
                    exp=None,
                    dropout_results=drop_res,
                    dropout_parameters=dropout_params,
                    prms={
                        "vals": [model_name],
                        "name": "ModelType",
                        "dataset": dataset_name,
                        "dropout": dropout_rate,
                        "lr": lr,
                        "weight_decay": 0
                    },
                    dropout_type="alignment-based",
                )

    except Exception as e:
        print(f"[DDP_RANK={rank}] ERROR => {e}")
        traceback.print_exc()
        raise e
    finally:
        dist.destroy_process_group()
        print(f"[DDP_RANK={rank}] Exiting rank {rank}...")

In [3]:
# Cell 3: The Main Entry (DDP or Non-DDP)
def run_experiment_ddp():
    """
    This function spawns multiple processes for DDP (one per GPU).
    """
    port = random.randint(20000, 30000)
    print(f"Launching {WORLD_SIZE} processes with tcp://127.0.0.1:{port} for DDP.")
    torch.multiprocessing.set_start_method("spawn", force=True)

    mp.spawn(
        ddp_worker,
        nprocs=WORLD_SIZE,
        args=(WORLD_SIZE, port, EPOCHS, BATCH_SIZE, LEARNING_RATE, DROPOUT_RATE),
        join=True
    )
    print("All DDP processes done successfully.")


def run_experiment_noddp():
    """
    This function runs a single-process, single-GPU version of the same logic,
    for quick debugging or smaller tests.
    """
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Running NON-DDP single-process on device={device}")

    net = get_model(
        model_name,
        build=True,
        dataset=dataset_name,
        dropout=DROPOUT_RATE,
        ignore_flag=False
    )
    net.to(device)

    loader_params = dict(batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    dataset = get_dataset(
        dataset_name,
        build=True,
        transform_parameters=net,
        loader_parameters=loader_params,
        device="cuda",
        distributed=False
    )

    optimizer = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE, weight_decay=0)
    train_params = dict(
        num_epochs=EPOCHS,
        alignment=True,
        alignment_expansion=False,
        compare_expected=False,
        frequency=1,
        delta_alignment=False,
    )
    nets = [net]
    optimizers = [optimizer]

    print(f"Starting single-process training for {EPOCHS} epochs...")
    results = train.train(nets, optimizers, dataset, **train_params)
    print(f"Done training. Results keys={list(results.keys())}")

    if "alignment" not in results:
        print("No 'alignment' => skipping dropout.")
    else:
        dropout_params = dict(
            num_drops=3,
            by_layer=False,
            train_set=False,
        )
        drop_res = progressive_dropout(nets, dataset, alignment=results["alignment"], **dropout_params)
        print("Dropout results keys:", list(drop_res.keys()))
        plotting.plot_dropout_results(
            exp=None,
            dropout_results=drop_res,
            dropout_parameters=dropout_params,
            prms={
                "vals": [model_name],
                "name": "ModelType",
                "dataset": dataset_name,
                "dropout": DROPOUT_RATE,
                "lr": LEARNING_RATE,
                "weight_decay": 0
            },
            dropout_type="alignment-based",
        )

In [4]:
# Cell 4: Actually Run
if USE_DDP:
    run_experiment_ddp()
else:
    run_experiment_noddp()

Launching 4 processes with tcp://127.0.0.1:23044 for DDP.


Traceback (most recent call last):
  File "<string>", line 1, in <module>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/n/home13/hsafaai/.conda/envs/networkAlignmentAnalysis/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
  File "/n/home13/hsafaai/.conda/envs/networkAlignmentAnalysis/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
  File "/n/home13/hsafaai/.conda/envs/networkAlignmentAnalysis/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
  File "/n/home13/hsafaai/.conda/envs/networkAlignmentAnalysis/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/n/home13/hsafaai/.conda/envs/networkAlignmentAnalysis/lib/python3.9/multiprocessing/spawn.py", line 126, in _main
    exitcode = _main(fd, pa

ProcessExitedException: process 3 terminated with exit code 1